# DR-GradCAM-Evaluation: Genuine Experimental Pipeline

This notebook executes the **genuine, non-destructive inference pipeline** for:
1. **IDRiD Domain Shift Diagnostic** ($N=413$ training images)
2. **Phase 6 Grad-CAM Lesion Localization** ($N=81$ segmentation images: 78 Primary + 3 Supplementary)
3. **Phase 7 Decision Faithfulness Evaluation** (Progressive deletion test with valid retinal FOV parity)

> **Backbone Checkpoint:** `experiments/checkpoints/best_model_qwk.pth` (Trained strictly on APTOS 2019, $N=2,929$)
> **Integrity Guarantees:** 100% frozen weights, zero optimizer updates, and dual immutability verification (SHA-256 + bit-exact `state_dict`).

### Cell 1: Mount Google Drive & Clone Repository

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Clone repository
!git clone https://github.com/Shrees1ngh/DR-GradCAM-Evaluation.git
%cd DR-GradCAM-Evaluation
!git pull origin main

### Cell 2: Install Required Dependencies

In [ ]:
!pip install -r requirements.txt

### Cell 3: Link Trained APTOS Checkpoint & IDRiD Dataset from Google Drive

In [ ]:
import os
import shutil

os.makedirs('experiments/checkpoints', exist_ok=True)

# Adjust DRIVE_CKPT to match your checkpoint path on Google Drive
DRIVE_CKPT = '/content/drive/MyDrive/DR-GradCAM-Evaluation/best_model_qwk.pth'
LOCAL_CKPT = 'experiments/checkpoints/best_model_qwk.pth'

if os.path.exists(DRIVE_CKPT):
    shutil.copy(DRIVE_CKPT, LOCAL_CKPT)
    print(f'✅ Checkpoint copied to {LOCAL_CKPT}')
else:
    print(f'⚠️ Checkpoint not found at {DRIVE_CKPT}. Please verify path!')

# Unzip raw IDRiD dataset into data/raw/ if packaged as zip
DRIVE_IDRID_ZIP = '/content/drive/MyDrive/DR-GradCAM-Evaluation/IDRiD.zip'
if os.path.exists(DRIVE_IDRID_ZIP):
    !unzip -q {DRIVE_IDRID_ZIP} -d data/raw/
    print('✅ IDRiD raw dataset unzipped to data/raw/')
else:
    print('ℹ️ Verify that data/raw/IDRiD directory is extracted.')

### Cell 4: Preprocess IDRiD Images & Generate Persistent FOV Masks

In [ ]:
!python scripts/preprocess_data.py --size 512 --idrid-only

### Cell 5: Run Genuine Domain Shift Diagnostic (N=413 IDRiD Train Images)

In [ ]:
!python scripts/diagnose_idrid_domain.py \
    --checkpoint experiments/checkpoints/best_model_qwk.pth \
    --idrid-manifest data/processed/idrid/external_train.csv

### Cell 6: Run Genuine Phase 6 Localization Evaluation (N=81 Images)

In [ ]:
!python scripts/evaluate_localization.py \
    --checkpoint experiments/checkpoints/best_model_qwk.pth \
    --manifest data/processed/idrid/segmentation_manifest.csv

### Cell 7: Run Genuine Phase 7 Faithfulness Evaluation (Deletion Test with FOV Parity)

In [ ]:
!python scripts/evaluate_faithfulness.py \
    --checkpoint experiments/checkpoints/best_model_qwk.pth \
    --manifest data/processed/idrid/segmentation_manifest.csv \
    --deletion-steps 0.10,0.20,0.30 \
    --fill-strategies black,gaussian_blur \
    --num-random-trials 5

### Cell 8: Sync Reports and Predictions Back to Google Drive

In [ ]:
DRIVE_REPORTS = '/content/drive/MyDrive/DR-GradCAM-Evaluation/reports'
os.makedirs(DRIVE_REPORTS, exist_ok=True)

!cp -r reports/* {DRIVE_REPORTS}/
print(f'✅ All reports and predictions successfully backed up to {DRIVE_REPORTS}!')

### Cell 9: Quick Inspection of Genuine Results

In [ ]:
import pandas as pd

if os.path.exists('reports/idrid_training_predictions.csv'):
    df_diag = pd.read_csv('reports/idrid_training_predictions.csv')
    print(f'Domain Diagnostic Predictions: {len(df_diag)} records')
    print(f'Accuracy: {(df_diag["true_grade"] == df_diag["predicted_grade"]).mean() * 100:.2f}%')
    display(df_diag.head(10))

if os.path.exists('reports/faithfulness_predictions.csv'):
    df_faith = pd.read_csv('reports/faithfulness_predictions.csv')
    print(f'Faithfulness Predictions: {len(df_faith)} records')
    display(df_faith.head(10))